# 07 — Final Results Aggregation (`07_results_table.ipynb`)

**Purpose:** Aggregate all `results/*.json` outputs across **Run A** (Clean ZWJ, $N=5{,}723$) and **Run B** (Emoji + Conversational Punctuation Kept, Gold-Safe Dedup, $N=5{,}834$) on the 300-row human gold benchmark into:
1. `results_table.csv` (and per-run CSVs `results_table_runA.csv`, `results_table_runB.csv`)
2. `results_table.tex` (clean LaTeX `tabular` using raw strings with bolded per-column best scores via `\textbf{...}` and `--` for N/A single-task cells)
3. `results_head_ablation.csv` & `results_head_ablation.tex` (when `05d_head_ablation_h*` JSON files exist in `results/`).

## 1. Environment & Single Config Cell

In [ ]:
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(
            ["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"],
            check=True,
        )
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import glob
import json
import numpy as np
import pandas as pd

from eval_utils import log_environment

# -- Single Config Cell -------------------------------------------------------
WORK_DIR     = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RESULTS_DIR  = os.path.join(WORK_DIR, "results")
OUT_CSV      = os.path.join(WORK_DIR, "results_table.csv")
OUT_TEX      = os.path.join(WORK_DIR, "results_table.tex")
OUT_H_CSV    = os.path.join(WORK_DIR, "results_head_ablation.csv")
OUT_H_TEX    = os.path.join(WORK_DIR, "results_head_ablation.tex")

env_info = log_environment()

## 2. Load All esults/*.json & Format Mean ± Std Table (Run A & Run B)

In [ ]:
ROW_SPEC = [
    ("Majority class",                     ["01_majority_baseline"]),
    ("TF-IDF + LR (word)",                 ["02_tfidf_lr_baseline_word"]),
    ("TF-IDF + LR (char)",                 ["02_tfidf_lr_baseline_char", "02_tfidf_lr_baseline"]),
    ("XLM-R plain (noMLM+mean)",           ["03_xlmr_plain_baseline", "ablation_noMLM+mean"]),
    ("noMLM+attn",                         ["ablation_noMLM+attn"]),
    ("MLM+mean",                           ["ablation_MLM+mean"]),
    ("MLM+attn (proposed)",                ["ablation_MLM+attn"]),
    ("Control X: genericMLM+attn",         ["04_matched_compute_mlm_control_genericMLM+attn", "04_matched_compute_mlm_control"]),
    ("Control Y1: noMLM+attn_highEncLR",   ["04_matched_compute_mlm_control_noMLM+attn_highEncLR"]),
    ("Control Y2: noMLM+attn_13ep",        ["04_matched_compute_mlm_control_noMLM+attn_13ep"]),
    ("Single-task sentiment (plain)",      ["05a_single_task_sentiment_plain"]),
    ("Single-task sentiment (MLM+attn)",   ["05a_single_task_sentiment_mlm_attn", "05_single_task_sentiment"]),
    ("Single-task intent (plain)",         ["05b_single_task_intent_plain"]),
    ("Single-task intent (MLM+attn)",      ["05b_single_task_intent_mlm_attn"]),
    ("Two-model pipeline (plain)",         ["05c_pipeline_2model_plain"]),
    ("Two-model pipeline (MLM+attn)",      ["05c_pipeline_2model_mlm_attn"]),
    ("Cascade: Sent -> Intent (MLM+attn)", ["05c_cascade_sent_to_intent"]),
    ("Cascade: Intent -> Sent (MLM+attn)", ["05c_cascade_intent_to_sent"]),
]

METRIC_COLS = [
    ("sent_acc",     "Sent Acc"),
    ("sent_f1_3cls", "Sent F1 (3-cls)"),
    ("sent_f1_4cls", "Sent F1 (4-cls)"),
    ("intent_acc",   "Intent Acc"),
    ("intent_f1",    "Intent F1"),
    ("joint_acc",    "Joint Acc"),
]

json_records = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json"))):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "notebook" in data and "run" in data:
        json_records.append(data)

print(f"Loaded {len(json_records)} result JSON files from `{RESULTS_DIR}/`.")

def select_seed_records(matches: list, nb_tags: list) -> list:
    """For the first matching notebook tag, prefer individual non-aggregated seed runs over _agg files."""
    for tag in nb_tags:
        tag_recs = [r for r in matches if r["notebook"] == tag]
        if not tag_recs:
            continue
        indiv = [r for r in tag_recs if not r.get("is_preaggregated", False)]
        chosen = indiv if indiv else tag_recs
        by_seed = {}
        for r in chosen:
            if r["seed"] not in by_seed:
                by_seed[r["seed"]] = r
        return [by_seed[k] for k in sorted(by_seed.keys(), key=str)]
    return []

def build_run_table(run_name: str):
    rows_raw = []
    for display_label, nb_tags in ROW_SPEC:
        matches = [r for r in json_records if r["run"] == run_name and r["notebook"] in nb_tags]
        seed_list = select_seed_records(matches, nb_tags)

        row_dict = {"Run": run_name, "Model": display_label, "n_seeds": len(seed_list)}
        for m_key, col_name in METRIC_COLS:
            vals = [x[m_key] for x in seed_list if x.get(m_key) is not None]
            if not vals:
                row_dict[f"{m_key}_mean"] = np.nan
                row_dict[f"{m_key}_std"]  = np.nan
                row_dict[col_name]        = "--"
            elif len(vals) == 1 and not seed_list[0].get("is_preaggregated", False):
                m = float(vals[0])
                row_dict[f"{m_key}_mean"] = m
                row_dict[f"{m_key}_std"]  = 0.0
                row_dict[col_name]        = f"{m:.3f}"
            else:
                m = float(np.mean(vals))
                s = float(seed_list[0].get(f"{m_key}_std", 0.0)) if len(vals) == 1 else float(np.std(vals, ddof=0))
                row_dict[f"{m_key}_mean"] = m
                row_dict[f"{m_key}_std"]  = s
                row_dict[col_name]        = f"{m:.3f} ± {s:.3f}" if s > 0 else f"{m:.3f}"
        rows_raw.append(row_dict)

    df_run = pd.DataFrame(rows_raw)
    best_indices = {}
    for m_key, _ in METRIC_COLS:
        valid = df_run[f"{m_key}_mean"].dropna()
        if len(valid) > 0:
            max_val = valid.max()
            best_indices[m_key] = set(df_run.index[np.isclose(df_run[f"{m_key}_mean"], max_val, atol=1e-4)])
        else:
            best_indices[m_key] = set()
    return df_run, best_indices

df_runA, best_A = build_run_table("runA")
df_runB, best_B = build_run_table("runB")

display_cols = ["Run", "Model"] + [c for _, c in METRIC_COLS]
print("=== Run A (Clean ZWJ, N=5,723) -- 300 Gold Test Rows ===")
display(df_runA[display_cols])

print("\n=== Run B (Emoji + Punctuation Kept, Gold-Safe Dedup, N=5,834) -- 300 Gold Test Rows ===")
display(df_runB[display_cols])

## 3. Export esults_table.csv, Clean LaTeX esults_table.tex, and Head-Count Table

In [ ]:
combined_df = pd.concat([df_runA[display_cols], df_runB[display_cols]], ignore_index=True)
combined_df.to_csv(OUT_CSV, index=False)
df_runA[display_cols].to_csv(os.path.join(WORK_DIR, "results_table_runA.csv"), index=False)
df_runB[display_cols].to_csv(os.path.join(WORK_DIR, "results_table_runB.csv"), index=False)
print(f"Exported {OUT_CSV}, results_table_runA.csv, and results_table_runB.csv")

def escape_tex_label(s: str) -> str:
    return (
        s.replace("_", r"\_")
         .replace("->", r"$\rightarrow$")
         .replace("&", r"\&")
    )

def to_latex_tabular(df_A, best_A, df_B, best_B) -> str:
    lines = [
        r"% Auto-generated by 07_results_table.ipynb",
        r"\begin{table*}[t]",
        r"\caption{Baseline, factorial ablation, single-task, two-model, and cascade controls evaluated on the 300-row human gold benchmark (mean $\pm$ std over 3 seeds: 42, 43, 44). Best value in each column per regime is shown in \textbf{bold}.}",
        r"\label{tab:full_baseline_ablation_results}",
        r"\centering",
        r"\footnotesize",
        r"\begin{tabular}{llcccccc}",
        r"\toprule",
        r"\textbf{Regime} & \textbf{Model} & \textbf{Sent Acc} & \textbf{Sent F1 (3-cls)} & \textbf{Sent F1 (4-cls)} & \textbf{Intent Acc} & \textbf{Intent F1} & \textbf{Joint Acc} \\",
        r"\midrule",
    ]

    for regime_label, df_r, best_r in [
        ("Run A (Clean ZWJ)", df_A, best_A),
        ("Run B (Emoji+Punct)", df_B, best_B),
    ]:
        for idx, row in df_r.iterrows():
            r_col = regime_label if idx == 0 else ""
            m_col = escape_tex_label(str(row["Model"]))
            cells = []
            for m_key, col_name in METRIC_COLS:
                val_str = str(row[col_name]).replace("±", r"$\pm$")
                if idx in best_r.get(m_key, set()) and val_str not in ("--", "—"):
                    val_str = r"\textbf{" + val_str + "}"
                cells.append(val_str)
            lines.append(f"{r_col} & {m_col} & " + " & ".join(cells) + r" \\")
        if regime_label.startswith("Run A"):
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table*}",
    ])
    return "\n".join(lines)

latex_str = to_latex_tabular(df_runA, best_A, df_runB, best_B)
with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex_str + "\n")

print(f"Exported clean LaTeX table to {OUT_TEX}:\n")
print(latex_str)

# Supplementary: Aggregate 05d attention head-count sweep if JSONs exist
head_recs = [r for r in json_records if str(r.get("notebook", "")).startswith("05d_head_ablation_h")]
if head_recs:
    head_rows = []
    for run in ("runA", "runB"):
        for h in (1, 2, 4, 8, 12):
            tag = f"05d_head_ablation_h{h}"
            sub = [r for r in head_recs if r["run"] == run and r["notebook"] == tag]
            if not sub:
                continue
            row = {"Run": run, "Heads": h, "Head Dim": 768 // h, "n_seeds": len(sub)}
            for m_key, col_name in METRIC_COLS:
                vals = [x[m_key] for x in sub if x.get(m_key) is not None]
                row[col_name] = f"{np.mean(vals):.3f} ± {np.std(vals, ddof=0):.3f}" if len(vals) > 1 else (f"{vals[0]:.3f}" if vals else "--")
            head_rows.append(row)
    if head_rows:
        df_h = pd.DataFrame(head_rows)
        df_h.to_csv(OUT_H_CSV, index=False)
        h_lines = [
            r"% Auto-generated head-count ablation table by 07_results_table.ipynb",
            r"\begin{table}[htbp]",
            r"\caption{Attention head-count ablation ($h \in \{1, 2, 4, 8, 12\}$) for Joint \texttt{MLM+attn}.}",
            r"\label{tab:head_ablation}",
            r"\centering\footnotesize",
            r"\begin{tabular}{lccccccc}",
            r"\toprule",
            r"\textbf{Run} & \textbf{Heads ($h$)} & \textbf{Sent Acc} & \textbf{Sent F1 (3-cls)} & \textbf{Sent F1 (4-cls)} & \textbf{Intent Acc} & \textbf{Intent F1} & \textbf{Joint Acc} \\",
            r"\midrule",
        ]
        for _, r in df_h.iterrows():
            vals = [str(r[c]).replace("±", r"$\pm$") for _, c in METRIC_COLS]
            h_lines.append(f"{r['Run']} & {r['Heads']} ($d_k={r['Head Dim']}$) & " + " & ".join(vals) + r" \\")
        h_lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
        with open(OUT_H_TEX, "w", encoding="utf-8") as f:
            f.write("\n".join(h_lines) + "\n")
        print(f"\nExported head-count ablation summary: {OUT_H_CSV} & {OUT_H_TEX}")
        display(df_h)